In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
#    for filename in filenames:
#        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------- settings ----------------
SRC_MANIFEST = Path("/kaggle/input/notebooks/t1f3001843/dsai-preprocessing/outputs/processed/chexpert_manifest.csv")
OUT_DIR = Path("/kaggle/working/stable_split")
EXTRA_VALID_PATIENTS = 6000   # must match the preprocessing notebook
VALID_SEED = 42               # must match the preprocessing notebook
LABELS = ["Pleural Effusion", "Cardiomegaly", "Pneumothorax", "Atelectasis", "Consolidation", "Lung Lesion"]
# ------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = pd.read_csv(SRC_MANIFEST)

if "split_origin" in manifest.columns:
    raise ValueError("Source manifest already has a stable split; refusing to carve a second time.")
missing = {"split", "patient_id", "shard_index", "shard_path", *LABELS} - set(manifest.columns)
if missing:
    raise ValueError(f"Manifest is missing columns: {sorted(missing)}")

manifest["split"] = manifest["split"].astype(str).str.lower().replace({"validation": "valid"})
manifest["patient_id"] = manifest["patient_id"].astype(str)
assert set(manifest["split"]) <= {"train", "valid"}, set(manifest["split"])
print(manifest["split"].value_counts())

split
train    191027
valid       202
Name: count, dtype: int64


In [3]:
train_patients = np.array(sorted(manifest.loc[manifest["split"].eq("train"), "patient_id"].unique()))
official_valid_patients = set(manifest.loc[manifest["split"].eq("valid"), "patient_id"])
assert official_valid_patients.isdisjoint(set(train_patients)), "official train/valid patients overlap"

rng = np.random.default_rng(VALID_SEED)
carved = set(rng.choice(train_patients, size=EXTRA_VALID_PATIENTS, replace=False))

manifest["split_origin"] = np.where(manifest["split"].eq("valid"), "official_valid", "official_train")
carve_mask = manifest["patient_id"].isin(carved)
manifest.loc[carve_mask, "split"] = "valid"
manifest.loc[carve_mask, "split_origin"] = "carved_from_train"

In [4]:
tr = manifest[manifest["split"].eq("train")]
va = manifest[manifest["split"].eq("valid")]

assert set(tr["patient_id"]).isdisjoint(set(va["patient_id"])), "Patient leakage"
assert not manifest.duplicated(["shard_path", "shard_index"]).any(), "Duplicate shard rows"

print(f"train: {len(tr):,} images / {tr['patient_id'].nunique():,} patients")
print(f"valid: {len(va):,} images / {va['patient_id'].nunique():,} patients")
print(manifest.groupby(["split", "split_origin"]).size())

def pos_rate(df):
    return (df[LABELS] == 1).mean().mul(100).round(2)

display(pd.DataFrame({
    "train_pos_%": pos_rate(tr),
    "valid_all_pos_%": pos_rate(va),
    "carved_pos_%": pos_rate(va[va["split_origin"].eq("carved_from_train")]),
    "official_pos_%": pos_rate(va[va["split_origin"].eq("official_valid")]),
}))

train: 172,924 images / 58,534 patients
valid: 18,305 images / 6,200 patients
split  split_origin     
train  official_train       172924
valid  carved_from_train     18103
       official_valid          202
dtype: int64


,train_pos_%,valid_all_pos_%,carved_pos_%,official_pos_%
Pleural Effusion,40.25,40.22,40.31,31.68
Cardiomegaly,12.23,12.60,12.38,32.67
Pneumothorax,9.22,9.57,9.63,3.47
Atelectasis,15.51,16.21,15.98,37.13
Consolidation,6.81,6.77,6.67,15.84
Lung Lesion,3.69,3.57,3.60,0.50


In [5]:
manifest.to_csv(OUT_DIR / "chexpert_manifest_stable.csv", index=False)
manifest.groupby("patient_id", as_index=False)[["split", "split_origin"]].first().to_csv(
    OUT_DIR / "split_assignment.csv", index=False)

with open(OUT_DIR / "split_config.json", "w") as f:
    json.dump({
        "source_manifest": str(SRC_MANIFEST),
        "extra_valid_patients": EXTRA_VALID_PATIENTS,
        "seed": VALID_SEED,
        "n_train_images": int(len(tr)), "n_valid_images": int(len(va)),
        "n_train_patients": int(tr["patient_id"].nunique()),
        "n_valid_patients": int(va["patient_id"].nunique()),
    }, f, indent=2)
print("Saved to", OUT_DIR)

Saved to /kaggle/working/stable_split
